In [9]:
from google.colab import files
uploaded = files.upload()

Saving Dataset for Data Analytics.xlsx to Dataset for Data Analytics.xlsx


In [4]:
!pip install openpyxl

In [12]:
# ==========================================================
# Project 1: Advanced EDA & Feature Engineering
# Dataset: Dataset_for_Data_Analytics.xlsx (E-commerce orders)
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------------------------------------
# STEP 0: Load the data
# -----------------------------------------------------------
df = pd.read_excel("/content/Dataset for Data Analytics.xlsx")

print("Shape of dataset:", df.shape)
print("\nColumn types:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())

Shape of dataset: (1200, 14)

Column types:
 OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object

First 5 rows:
      OrderID       Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200000 2023-01-04     C72649  Monitor         5     570.62   
1  ORD200001 2024-08-23     C75739    Phone         2     151.35   
2  ORD200002 2024-02-27     C81728   Tablet         5     550.68   
3  ORD200003 2023-10-15     C33540    Chair         1     273.19   
4  ORD200004 2025-05-08     C81840  Printer         4     626.01   

  ShippingAddress PaymentMethod OrderSta

In [13]:
# STEP 2: Detect and handle outliers (IQR method)
# -----------------------------------------------------------
numeric_cols = ["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]

def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

print("\nOutlier check (IQR method):")
for col in numeric_cols:
    lower, upper = iqr_bounds(df[col])
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: bounds=({lower:.2f}, {upper:.2f}), outliers={n_outliers}")

# Visualize outliers with boxplots before capping
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(16, 4))
for ax, col in zip(axes, numeric_cols):
    ax.boxplot(df[col])
    ax.set_title(col)
plt.tight_layout()
plt.savefig("boxplots_before_capping.png")
plt.close()

# Cap (winsorize) TotalPrice outliers instead of deleting rows,
# since deleting rows would shrink the dataset for no good reason.
lower, upper = iqr_bounds(df["TotalPrice"])
df["TotalPrice"] = df["TotalPrice"].clip(lower, upper)

print("\nTotalPrice stats after capping:\n", df["TotalPrice"].describe())



Outlier check (IQR method):
Quantity: bounds=(-1.00, 7.00), outliers=0
UnitPrice: bounds=(-317.20, 1024.83), outliers=0
ItemsInCart: bounds=(-0.50, 11.50), outliers=0
TotalPrice: bounds=(-1341.41, 3330.41), outliers=8

TotalPrice stats after capping:
 count    1200.000000
mean     1053.643183
std       818.937858
min        11.390000
25%       410.520000
50%       823.615000
75%      1578.475000
max      3330.407500
Name: TotalPrice, dtype: float64


In [14]:
# -----------------------------------------------------------
# STEP 3: Feature Engineering (3+ new predictive features)
# -----------------------------------------------------------

# Feature 1: Month the order was placed (captures seasonality)
df["OrderMonth"] = df["Date"].dt.month

# Feature 2: Average value per item in the cart
# (helps distinguish "few expensive items" vs "many cheap items")
df["AvgItemValue"] = df["TotalPrice"] / df["ItemsInCart"]

# Feature 3: Binary flag - did the customer use a real coupon?
df["HasCoupon"] = (df["CouponCode"] != "NoCoupon").astype(int)

# Feature 4 (bonus): Cart fill rate - how much of the cart was
# actually bought vs just browsed/added
df["CartFillRate"] = df["Quantity"] / df["ItemsInCart"]

print("\nNew feature preview:\n",
      df[["OrderMonth", "AvgItemValue", "HasCoupon", "CartFillRate"]].head())



New feature preview:
    OrderMonth  AvgItemValue  HasCoupon  CartFillRate
0           1    407.585714          1      0.714286
1           8    100.900000          1      0.666667
2           2    344.175000          1      0.625000
3          10     54.638000          1      0.200000
4           5    313.005000          1      0.500000


In [15]:
# -----------------------------------------------------------
# STEP 4: Encode categorical columns (One-Hot Encoding)
# -----------------------------------------------------------
# One-hot encoding avoids implying a false numeric order between
# categories (e.g. Online != 2x Cash), unlike label encoding.
categorical_cols = ["PaymentMethod", "OrderStatus", "ReferralSource", "Product"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("\nShape after one-hot encoding:", df_encoded.shape)



Shape after one-hot encoding: (1200, 32)


In [16]:
# -----------------------------------------------------------
# STEP 5: Save the cleaned dataset
# -----------------------------------------------------------
df.to_csv("cleaned_dataset.csv", index=False)
df_encoded.to_csv("cleaned_dataset_encoded.csv", index=False)

print("\nDone. Files saved: cleaned_dataset.csv, cleaned_dataset_encoded.csv")


Done. Files saved: cleaned_dataset.csv, cleaned_dataset_encoded.csv
